In [26]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.integrate import quad
import torch

from copy import deepcopy

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import VecMonitor

### Add mbt-gym to path

In [27]:
import sys
sys.path.append("../../")

In [28]:
from mbt_gym.gym.ModelDynamics import LimitOrderModelDynamics
from mbt_gym.agents.BaselineAgents import *
from mbt_gym.stochastic_processes.midprice_models import *
from mbt_gym.stochastic_processes.arrival_models import *
from mbt_gym.stochastic_processes.fill_probability_models import *

# helpers
from mbt_gym.gym.TradingEnvironment import TradingEnvironment
from mbt_gym.gym.StableBaselinesTradingEnvironment import StableBaselinesTradingEnvironment
from mbt_gym.gym.index_names import *
from mbt_gym.gym.helpers.plotting import *
from mbt_gym.gym.helpers.generate_trajectory import generate_trajectory
from mbt_gym.gym.wrappers import *

In [29]:
seed = 42

max_inventory = 20

initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10
phi = 15
k = 1
fill_exponent = k
gamma = k
alpha=0.001
mu=0
big_phi=0.1

### Function for building the enviroment

In [30]:
def get_as_env(num_trajectories:int = 1, fads_proportion:float=0.6, 
               psi:float = 15.0, phi:float = 15.0, eta:float = 10.0,
               gamma:float = 1.0,
               ) -> TradingEnvironment:
    midprice_model = ArithmeticBrownianMotionWithFadsMidpriceModel(initial_price=initial_price, drift=mu,
                                                 volatility=sigma, fads_proportion=fads_proportion, eta=eta, step_size=terminal_time/n_steps,
                                                 terminal_time=terminal_time,
                                                 num_trajectories=num_trajectories, seed=seed)
    arrival_model = ModifiedPoissonArrivalModel(phi=phi,
                                                psi=psi,
                                                gamma=gamma,
                                                fads_proportion=fads_proportion,
                                                sigma=sigma,
                                                step_size=terminal_time/n_steps,
                                                num_trajectories=num_trajectories,
                                                seed=seed)
    fill_probability_model = ExponentialFillFunction(fill_exponent=k, 
                                                     step_size=terminal_time/n_steps,
                                                     num_trajectories=num_trajectories,
                                                     seed=seed)
    LOtrader = LimitOrderModelDynamics(midprice_model = midprice_model, arrival_model = arrival_model, 
                                fill_probability_model = fill_probability_model,
                                num_trajectories = num_trajectories, seed=seed)
    reward = CjMmCriterion(per_step_inventory_aversion = big_phi,
                           terminal_inventory_aversion = alpha,
                           terminal_time = terminal_time)
    env_params = dict(terminal_time=terminal_time, 
                      n_steps=n_steps,
                      seed = seed,
                      initial_inventory = initial_inventory,
                      model_dynamics = LOtrader,
                      reward_function = reward,
                      max_inventory=max_inventory,
                      normalise_action_space = False,
                      normalise_observation_space = False,
                      num_trajectories=num_trajectories)
    return TradingEnvironment(**env_params)

## Varying fad proportion (paramter q)

### Parameters

In [31]:
result_list = []
num_trajectories=10000

seed = 42

In [32]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1]
# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1
max_inventory=100

In [33]:
# OU parameters
u0 = 0.0
xi = 1.0
total_arrivals = 30.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(gamma, fads_proportion_values):
    """Compute psi for multiple eta values using the discretized integral version."""
    time_grid = np.linspace(0, terminal_time, n_steps)
    psi_results = {}

    for fads_proportion in fads_proportion_values:
        v_t = v(time_grid, eta)
        c = gamma * sigma * fads_proportion
        integrand_values = np.exp((c**2) / 2 * v_t)
        aux_integral = terminal_time * np.mean(integrand_values)
        psi = (total_arrivals * terminal_time - phi * terminal_time) / aux_integral
        psi_results[fads_proportion] = psi

    return psi_results

psi_values = compute_psi(gamma, fads_proportion_values)

for fads_proportion, val in psi_values.items():
    print(f"fads_proportion={fads_proportion:.2f} -> psi={val:.6f}")

fads_proportion=0.00 -> psi=15.000000
fads_proportion=0.20 -> psi=14.985763
fads_proportion=0.40 -> psi=14.943132
fads_proportion=0.60 -> psi=14.872343
fads_proportion=0.80 -> psi=14.773788
fads_proportion=1.00 -> psi=14.648008


In [34]:
results_dict = {}
for q in fads_proportion_values:
    vec_env = get_as_env(
        num_trajectories=num_trajectories,
        fads_proportion=q,
        psi=psi_values[q]
    )

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

    #observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    _, _, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )
    del rewards, total_rewards

    results_dict[q] = dict(
        results=results,
        #rewards=total_rewards,
        #obs=observations
    )
result_list.append({'fads': results_dict})  

✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points


/Users/andrea/Desktop/gitProject/mbt_gym/Experiments_MMwithFadsInfUninfTraders/Performances_table/../../mbt_gym/agents/BaselineAgents.py:700: UserWarning: MM agent is quoting a negative spread
  warnings.warn("MM agent is quoting a negative spread")


In [35]:
header = f"{'Fads Prop':>10} | {'Psi':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for fads_prop, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{fads_prop:10.2f} | {psi_values[fads_prop]:10.4f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15.2f} | {std_inv:13.2f}")


 Fads Prop |        Psi |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
-----------------------------------------------------------------------------------
      0.00 |    15.0000 |      21.41 |       5.03 |           -0.02 |          3.13
      0.20 |    14.9858 |      21.40 |       5.02 |           -0.02 |          3.13
      0.40 |    14.9431 |      21.35 |       4.98 |           -0.02 |          3.16
      0.60 |    14.8723 |      21.25 |       4.91 |           -0.01 |          3.19
      0.80 |    14.7738 |      21.17 |       4.78 |           -0.00 |          3.23
      1.00 |    14.6480 |      21.32 |       4.60 |           -0.01 |          3.17


## Varying eta

### Parameters

In [36]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0

fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta_values = [2.5, 5, 7.5, 10.0, 12.5]
phi = 15
k = 1
gamma = k
alpha=0.001
mu=0
big_phi=0.1

In [37]:
# OU parameters
u0 = 0.0
xi = 1.0
total_arrivals = 30.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(gamma, eta_values):
    """Compute psi for multiple eta values using the discretized integral version."""
    time_grid = np.linspace(0, terminal_time, n_steps)
    psi_results = {}

    for eta in eta_values:
        v_t = v(time_grid, eta)
        c = gamma * sigma * fads_proportion
        integrand_values = np.exp((c**2) / 2 * v_t)
        aux_integral = terminal_time * np.mean(integrand_values)
        psi = (total_arrivals * terminal_time - phi * terminal_time) / aux_integral
        psi_results[eta] = psi

    return psi_results

psi_values = compute_psi(gamma, eta_values)

for eta, val in psi_values.items():
    print(f"eta={eta:.2f} -> psi={val:.6f}")

eta=2.50 -> psi=14.573043
eta=5.00 -> psi=14.758967
eta=7.50 -> psi=14.832983
eta=10.00 -> psi=14.872343
eta=12.50 -> psi=14.896720


In [38]:
results_dict = {}
for eta in eta_values:
    vec_env = get_as_env(num_trajectories=num_trajectories, eta=eta, psi=psi_values[eta])

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

        #observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    _, _, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )
    del rewards, total_rewards

    results_dict[eta] = dict(
        results=results,
        #rewards=total_rewards,
        #obs=observations
    )

result_list.append({'eta': results_dict})  

✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points


In [39]:
header = f"{'Eta':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for etas, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{etas:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

       Eta |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
       2.5 |      21.09 |       4.99 |         -0.0091 | 3.392877420420608
         5 |      21.17 |       4.94 |         -0.0138 | 3.256871130394938
       7.5 |      21.22 |       4.92 |         -0.0138 | 3.2149353897084776
      10.0 |      21.25 |       4.91 |         -0.0089 | 3.187604239864165
      12.5 |      21.28 |       4.90 |         -0.0126 | 3.1747190804857053


## Varying gamma parameter

### Parameters

In [40]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion = 0.6 # p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
phi = 15
k = 1
gamma_values = [0, 1, 2, 3]
alpha=0.001
mu=0
big_phi=0.1

In [41]:
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_aux_integral_for_psi(eta, gamma_values):
    """Compute psi for multiple gamma values using the discretized integral version."""
    time_grid = np.linspace(0, terminal_time, n_steps)
    v_t = v(time_grid, eta)
    
    psi_results = {}
    
    for gamma in gamma_values:
        c = gamma * sigma * fads_proportion
        integrand_values = np.exp((c**2) / 2 * v_t)
        aux_integral = terminal_time * np.mean(integrand_values)
        psi = (total_arrivals * terminal_time - phi * terminal_time) / aux_integral
        psi_results[gamma] = psi
    
    return psi_results

psi_values = compute_aux_integral_for_psi(eta, gamma_values)

for gamma, val in psi_values.items():
    print(f"gamma={gamma} -> psi={val:.6f}")

gamma=0 -> psi=15.000000
gamma=1 -> psi=14.872343
gamma=2 -> psi=14.495695
gamma=3 -> psi=13.888522


In [42]:
# OU parameters
u0 = 0.0
xi = 1.0

def m(t, eta):
    return u0 * np.exp(-eta * t)

def v(t, eta):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(q, eta, gamma):
    c = gamma * sigma * q
    def integrand(t):
        return np.exp(c * m(t, eta) + 0.5 * (c**2) * v(t, eta))
    denom, _ = quad(integrand, 0.0, terminal_time)
    return (30.0 - phi * terminal_time) / denom

# compute psi for each gamma
psi_values = {gamma: compute_psi(fads_proportion, eta, gamma) for gamma in gamma_values}

# pretty print
for gamma, val in psi_values.items():
    print(f"gamma={gamma} -> psi={val:.6f}")

gamma=0 -> psi=15.000000
gamma=1 -> psi=14.872283
gamma=2 -> psi=14.495463
gamma=3 -> psi=13.888033


In [43]:
results_dict = {}
for gamma in gamma_values:
    vec_env = get_as_env(num_trajectories=num_trajectories, gamma=gamma, psi=psi_values[gamma])

    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)

    #observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    _, _, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )
    del rewards, total_rewards

    results_dict[gamma] = dict(results=results, 
                               #rewards=total_rewards, 
                               #obs=observations
                               )
result_list.append({'gamma': results_dict})  

✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points


In [44]:
header = f"{'Gamma':>10} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))
for gamma, result in results_dict.items():
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    print(f"{gamma:10} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

     Gamma |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
----------------------------------------------------------------------
         0 |      21.41 |       4.90 |         -0.0187 | 3.1483885258970186
         1 |      21.25 |       4.91 |          -0.009 | 3.18752552930953
         2 |      21.08 |       4.91 |         -0.0087 | 3.2625793952025135
         3 |      20.87 |       4.91 |          0.0049 | 3.4011874382338885


## Varying Informed trader proportion (psi and phi)

### Parameters

In [45]:
initial_price = 100
terminal_time = 1.0
sigma = 1.0
n_steps = 1000
initial_inventory = 0
fads_proportion = 0.6

# p is defined directly inside the ArithmeticBrownianMotionWithFadsMidpriceModel class as sqrt(1-q^2)
eta = 10.0
k = 1
gamma = 1
alpha=0.001
mu=0
big_phi=0.1

In [46]:
# OU parameters
u0 = 0.0
xi = 1.0
total_arrivals = 30.0

def m(t):
    return u0 * np.exp(-eta * t)

def v(t):
    return (xi**2) / (2.0 * eta) * (1.0 - np.exp(-2.0 * eta * t))

def compute_psi(phi):
    """Compute psi for multiple eta values using the discretized integral version."""
    time_grid = np.linspace(0, terminal_time, n_steps)

    v_t = v(time_grid)
    c = gamma * sigma * fads_proportion
    integrand_values = np.exp((c**2) / 2 * v_t)
    aux_integral = terminal_time * np.mean(integrand_values)
    psi = (total_arrivals * terminal_time - phi * terminal_time) / aux_integral

    return psi

def phi_from_informed(perc_informed):
    """
    Given the percentage of informed traders (0-100),
    compute phi according to the paper.
    """
    return 30 * (1 - perc_informed / 100.0)

def psi_from_informed(perc_informed):
    """
    Given percentage of informed traders, compute phi and psi.
    """
    phi = phi_from_informed(perc_informed)
    psi = compute_psi(phi)  # eq (61) implementation

    return phi, psi

In [47]:
def build_phi_psi_dict(percentages):
    results = {}
    for perc in percentages:
        phi, psi = psi_from_informed(perc)
        results[perc] = {"phi": phi, "psi": psi}
    return results

percentages = [0, 25, 50, 75, 100]
phi_psi_dict = build_phi_psi_dict(percentages)
for perc, vals in phi_psi_dict.items():
    print(f"{perc}% informed → phi = {vals['phi']:.2f}, psi = {vals['psi']:.4f}")



0% informed → phi = 30.00, psi = 0.0000
25% informed → phi = 22.50, psi = 7.4362
50% informed → phi = 15.00, psi = 14.8723
75% informed → phi = 7.50, psi = 22.3085
100% informed → phi = 0.00, psi = 29.7447


In [48]:
results_dict = {}
for perc, vals in phi_psi_dict.items():
    phi, psi = vals['phi'], vals['psi']
    
    # Set up environment and agent
    vec_env = get_as_env(num_trajectories=num_trajectories, phi=phi, psi=psi)
    vec_as = OptimizedPartialInfoMMwithFadsInformedUniformedTradersAgent(env=vec_env)
    
    # Generate trajectory and results
        #observations, actions, rewards = generate_trajectory(vec_env, vec_as)
    _, _, rewards = generate_trajectory(vec_env, vec_as)
    results, _, total_rewards = generate_results_table_and_hist(
        vec_env=vec_env, agent=vec_as
    )
    del rewards, total_rewards
    
    # Store in results dictionary keyed by percentage
    results_dict[perc] = {
        "results": results,
        #"rewards": total_rewards,
        #"obs": observations
    }
result_list.append({'informed': results_dict})  

✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points
✓ Pre-computed ODE solutions for 1000 time points


In [49]:
header = f"{'Perc':>6} | {'Phi':>8} | {'Psi':>8} | {'Mean Spread':>11} | {'Mean PnL':>10} | {'Std PnL':>10} | {'Mean Term Inv':>15} | {'Std Term Inv':>13}"
print(header)
print("-" * len(header))

for perc, result in results_dict.items():
    phi, psi = phi_psi_dict[perc]['phi'], phi_psi_dict[perc]['psi']
    mean_spread = result['results'].loc['Inventory', 'Mean spread']
    mean_pnl = result['results'].loc['Inventory', 'Mean PnL']
    std_pnl = result['results'].loc['Inventory', 'Std PnL']
    mean_inv = result['results'].loc['Inventory', 'Mean terminal inventory']
    std_inv = result['results'].loc['Inventory', 'Std terminal inventory']
    
    print(f"{perc:6}% | {phi:8.2f} | {psi:8.4f} | {mean_spread:11.5f} | {mean_pnl:10.2f} | {std_pnl:10.2f} | {mean_inv:15} | {std_inv:13}")

  Perc |      Phi |      Psi | Mean Spread |   Mean PnL |    Std PnL |   Mean Term Inv |  Std Term Inv
------------------------------------------------------------------------------------------------------
     0% |    30.00 |   0.0000 |     2.06535 |      21.41 |       4.90 |         -0.0187 | 3.1483885258970186
    25% |    22.50 |   7.4362 |     2.06539 |      21.33 |       4.90 |         -0.0169 | 3.1538570655627374
    50% |    15.00 |  14.8723 |     2.06543 |      21.25 |       4.91 |         -0.0089 | 3.187604239864165
    75% |     7.50 |  22.3085 |     2.06547 |      21.16 |       4.91 |         -0.0085 | 3.2222395550300105
   100% |     0.00 |  29.7447 |     2.06551 |      21.07 |       4.92 |         -0.0111 | 3.2698588333443386


### Final Summary Table

In [50]:
fads_results = []
eta_results = []
gamma_results = []
informed_results = []

# Extract results from each experiment
for experiment_dict in result_list:
    for exp_type, results_dict in experiment_dict.items():
        for param, data in results_dict.items():
            results = data['results'].loc['Inventory']
            
            if exp_type == 'fads':
                fads_results.append({
                    'Type': 'Fads',
                    'Parameter': f'q={param:.1f}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'eta':
                eta_results.append({
                    'Type': 'Eta',
                    'Parameter': f'η={param:.1f}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'gamma':
                gamma_results.append({
                    'Type': 'Gamma',
                    'Parameter': f'γ={param}',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })
            elif exp_type == 'informed':
                informed_results.append({
                    'Type': 'Informed',
                    'Parameter': f'perc={param}%',
                    'Mean PnL': results['Mean PnL'],
                    'Std PnL': results['Std PnL'],
                    'Mean Term Inv': results['Mean terminal inventory'],
                    'Std Term Inv': results['Std terminal inventory']
                })

print("\nFINAL COMPLETE RESULTS SUMMARY")
print("=" * 100)

if fads_results:
    print("\nFADS PROPORTION EXPERIMENTS")
    print("-" * 100)
    df_fads = pd.DataFrame(fads_results)
    print(df_fads.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if eta_results:
    print("\nETA EXPERIMENTS")
    print("-" * 100)
    df_eta = pd.DataFrame(eta_results)
    print(df_eta.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if gamma_results:
    print("\nGAMMA EXPERIMENTS")
    print("-" * 100)
    df_gamma = pd.DataFrame(gamma_results)
    print(df_gamma.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

if informed_results:
    print("\nINFORMED TRADER PROPORTION EXPERIMENTS")
    print("-" * 100)
    df_informed = pd.DataFrame(informed_results)
    print(df_informed.to_string(index=False, float_format=lambda x: '{:.2f}'.format(x)))

print("\n" + "=" * 100)


FINAL COMPLETE RESULTS SUMMARY

FADS PROPORTION EXPERIMENTS
----------------------------------------------------------------------------------------------------
Type Parameter  Mean PnL  Std PnL  Mean Term Inv  Std Term Inv
Fads     q=0.0     21.41     5.03          -0.02          3.13
Fads     q=0.2     21.40     5.02          -0.02          3.13
Fads     q=0.4     21.35     4.98          -0.02          3.16
Fads     q=0.6     21.25     4.91          -0.01          3.19
Fads     q=0.8     21.17     4.78          -0.00          3.23
Fads     q=1.0     21.32     4.60          -0.01          3.17

ETA EXPERIMENTS
----------------------------------------------------------------------------------------------------
Type Parameter  Mean PnL  Std PnL  Mean Term Inv  Std Term Inv
 Eta     η=2.5     21.09     4.99          -0.01          3.39
 Eta     η=5.0     21.17     4.94          -0.01          3.26
 Eta     η=7.5     21.22     4.92          -0.01          3.21
 Eta    η=10.0     21.25   